# CVM-A1: phân tích thời gian và bộ nhớ

Notebook là giao diện xem dữ liệu P0 và gọi pipeline thực nghiệm. Chế độ mặc định đọc bộ P0 Linux có manifest; `SMOKE_LINUX` và `CUSTOM_SWEEP` gọi Runner API của TV3 để chạy C++ thật. API TV4 xác thực RAW/sidecar, tổng hợp bảng và tạo hình.

Run All mặc định chỉ đọc dữ liệu đã lưu, không khởi chạy benchmark. P0 gồm 135 lượt thuộc 15 cấu hình. Kết quả mô tả các cấu hình đã chạy; suy luận về độ phức tạp tiệm cận cần lập luận từ thuật toán và mô hình chi phí riêng.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from urllib.parse import quote

import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc có pyproject.toml và src/.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_HOME = Path(os.environ.get("XDG_DATA_HOME", str(Path.home() / ".local" / "share")))
EXPORT_DIR = Path(
    os.environ.get(
        "BIG_O_EXPORT_DIR",
        str(DATA_HOME / "distinct-elements-cvm-analysis" / "exports"),
    )
).expanduser()


def current_environment() -> dict:
    compiler = shutil.which("g++") or shutil.which("clang++")
    compiler_version = "không tìm thấy GCC/Clang"
    if compiler:
        result = subprocess.run(
            [compiler, "--version"], capture_output=True, text=True, check=False
        )
        compiler_version = (result.stdout or result.stderr).splitlines()[0]
    packages = {}
    for distribution in ("numpy", "scipy", "matplotlib", "pandas", "jupyterlab", "ipykernel"):
        try:
            packages[distribution] = version(distribution)
        except PackageNotFoundError:
            packages[distribution] = "chưa cài"
    return {
        "kernel": sys.executable,
        "python": platform.python_version(),
        "os": platform.platform(),
        "compiler": compiler_version,
        "packages": packages,
    }


ENVIRONMENT = current_environment()
environment_lines = []
for key, value in ENVIRONMENT.items():
    formatted = json.dumps(value, ensure_ascii=False) if isinstance(value, dict) else value
    environment_lines.append(f"**{key}** `{formatted}`")
display(Markdown("\n".join(environment_lines)))

## Chọn chế độ

Chọn `MODE` trong ô cấu hình:

- `READ_EXISTING_P0`: đọc bộ dữ liệu Linux theo manifest, xác thực 15 cặp RAW/sidecar và hiển thị bảng cùng ba hình TV4. Đây là chế độ mặc định; Run All không chạy C++.
- `SMOKE_LINUX`: chạy preset kiểm tra pipeline thật qua API TV3, sau đó phân tích qua API TV4.
- `CUSTOM_SWEEP`: xem trước hoặc chạy cấu hình. Để chạy C++ thật, đặt `RUN_CUSTOM = True`; profile `TV4_P0` dùng ma trận trong kế hoạch dự án, profile `MANUAL` dùng các tham số tùy chỉnh.

Các phép đo mới chỉ chạy khi bộ kiểm tra xác nhận Ubuntu/Linux và không phát hiện WSL, máy ảo hoặc container.

In [ ]:
MODE = "READ_EXISTING_P0"  # READ_EXISTING_P0 | SMOKE_LINUX | CUSTOM_SWEEP
RUN_CUSTOM = False
CUSTOM_PROFILE = "TV4_P0"  # TV4_P0 | MANUAL
READ_EXISTING_MANIFEST = ROOT / "notebooks" / "data" / "linux_p0" / "manifest.json"

# Chỉ dùng các giá trị này khi CUSTOM_PROFILE = "MANUAL".
CUSTOM_M = [256]
CUSTOM_N = 65536
CUSTOM_F0 = None
CUSTOM_EPSILON = 0.5
CUSTOM_DELTA = 0.05
CUSTOM_WORKLOADS = ["all_distinct"]
CUSTOM_SEED_GENERATOR = 44000
CUSTOM_SEED_ALGORITHM = 84000
CUSTOM_TRIALS = 2

ALLOWED_MODES = {"READ_EXISTING_P0", "SMOKE_LINUX", "CUSTOM_SWEEP"}
if MODE not in ALLOWED_MODES:
    raise ValueError(f"MODE phải thuộc {sorted(ALLOWED_MODES)}, nhận được {MODE!r}.")
if CUSTOM_PROFILE not in {"TV4_P0", "MANUAL"}:
    raise ValueError("CUSTOM_PROFILE phải là TV4_P0 hoặc MANUAL.")


def validate_custom_config() -> None:
    u64_max = 2**64 - 1
    if not CUSTOM_M or any(
        isinstance(m, bool) or not isinstance(m, int) or not 1 <= m <= u64_max for m in CUSTOM_M
    ):
        raise ValueError("CUSTOM_M phải là danh sách số nguyên uint64 dương, không rỗng.")
    if len(set(CUSTOM_M)) != len(CUSTOM_M):
        raise ValueError("CUSTOM_M không được chứa giá trị lặp.")
    allowed_workloads = {"all_duplicate", "all_distinct", "uniform", "skewed", "zipf"}
    if not CUSTOM_WORKLOADS or len(set(CUSTOM_WORKLOADS)) != len(CUSTOM_WORKLOADS):
        raise ValueError("CUSTOM_WORKLOADS phải không rỗng và không lặp.")
    if any(workload not in allowed_workloads for workload in CUSTOM_WORKLOADS):
        raise ValueError(f"Workload phải thuộc {sorted(allowed_workloads)}.")
    for name, value in (("epsilon", CUSTOM_EPSILON), ("delta", CUSTOM_DELTA)):
        if not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 < value < 1:
            raise ValueError(f"{name} phải hữu hạn và thuộc (0, 1).")
    if isinstance(CUSTOM_TRIALS, bool) or not isinstance(CUSTOM_TRIALS, int) or CUSTOM_TRIALS < 1:
        raise ValueError("CUSTOM_TRIALS phải là số nguyên dương.")
    seeds = (CUSTOM_SEED_GENERATOR, CUSTOM_SEED_ALGORITHM)
    if any(
        isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed <= u64_max
        for seed in seeds
    ):
        raise ValueError("Seed phải thuộc miền uint64.")
    if max(seeds) + CUSTOM_TRIALS - 1 > u64_max:
        raise ValueError("Lịch seed vượt miền uint64 khi xét số trial.")
    if CUSTOM_N is not None and (
        isinstance(CUSTOM_N, bool) or not isinstance(CUSTOM_N, int) or not 1 <= CUSTOM_N <= 2**64
    ):
        raise ValueError("CUSTOM_N phải thuộc [1, 2**64] hoặc là None.")
    if CUSTOM_F0 is not None and (
        isinstance(CUSTOM_F0, bool)
        or not isinstance(CUSTOM_F0, int)
        or not 1 <= CUSTOM_F0 <= u64_max
    ):
        raise ValueError("CUSTOM_F0 phải là uint64 dương hoặc là None.")
    if CUSTOM_N is not None and CUSTOM_F0 is not None and CUSTOM_F0 > CUSTOM_N:
        raise ValueError("CUSTOM_F0 không được lớn hơn CUSTOM_N.")
    for m in CUSTOM_M:
        if CUSTOM_F0 is not None and CUSTOM_F0 > m:
            raise ValueError(f"F0={CUSTOM_F0} vượt m={m}.")
        for workload in CUSTOM_WORKLOADS:
            if workload == "all_distinct":
                if CUSTOM_N is not None and CUSTOM_N < m:
                    raise ValueError(f"all_distinct yêu cầu n >= m (m={m}, n={CUSTOM_N}).")
                if CUSTOM_F0 is not None and CUSTOM_F0 != m:
                    raise ValueError("all_distinct yêu cầu F0=m.")
            if workload == "all_duplicate" and CUSTOM_F0 not in (None, 1):
                raise ValueError("all_duplicate yêu cầu F0=1.")


if MODE == "CUSTOM_SWEEP" and CUSTOM_PROFILE == "MANUAL":
    validate_custom_config()

## Cơ sở lựa chọn ma trận P0

Notebook đọc ma trận từ [kế hoạch benchmark](../experiments/benchmark_plan_timespace.csv); các tham số không được khai báo lặp trong giao diện.

- `m` tăng theo hệ số 4, tạo các mức cách đều trên thang log₂ và bao phủ hệ số 256 từ mức nhỏ nhất đến lớn nhất.
- `n` bằng mức `m` lớn nhất để `all_distinct` có miền ID đủ rộng ở mọi cấu hình.
- `F₀` được xác định theo workload: `1` cho `all_duplicate`, `m/4` cho `uniform`, và `m` cho `all_distinct`. Vì vậy so sánh giữa workload phản ánh đồng thời phân bố và mức `F₀`; thiết kế này không tách riêng hai yếu tố.
- `ε` và `δ` giữ cố định theo giao thức P0. `δ` là tham số thất bại của thuật toán, không phải mức tin cậy của khoảng thống kê cho runtime. Ngưỡng `s` lấy từ hàm trong mã nguồn; tỷ số `F₀/s` được tính để xem tải của từng cấu hình so với ngưỡng.
- Chín lượt và lịch seed là một phần của giao thức P0. Kết quả được diễn giải mô tả; số lượt này không tự bảo đảm độ chính xác của ước lượng thống kê. Thứ tự cấu hình được xáo trộn bằng seed do bộ lập kế hoạch TV4 cung cấp.

In [ ]:
from scripts.generate_timespace_plan import ORDER_SEED
from src.types import compute_cvm_threshold


def tv4_p0_configurations() -> list[dict]:
    plan_path = ROOT / "experiments" / "benchmark_plan_timespace.csv"
    with plan_path.open(encoding="utf-8", newline="") as stream:
        rows = [row for row in csv.DictReader(stream) if row["priority"] == "P0"]
    if len(rows) != 15:
        raise ValueError(f"Kế hoạch P0 cần 15 cấu hình, nhận {len(rows)}.")

    configs = []
    for row in rows:
        config = {
            "workload": row["workload"],
            "m": int(row["m"]),
            "n": int(row["n"]),
            "f0": int(row["f0"]),
            "epsilon": float(row["epsilon"]),
            "delta": float(row["delta"]),
            "trials": int(row["trials"]),
            "first_trial": int(row["first_trial"]),
            "seed_generator": int(row["seed_generator"]),
            "seed_algorithm": int(row["seed_algorithm"]),
            "s": int(row["s_reference"]),
        }
        expected_f0 = {
            "all_duplicate": 1,
            "uniform": config["m"] // 4,
            "all_distinct": config["m"],
        }.get(config["workload"])
        if expected_f0 is None or config["f0"] != expected_f0:
            raise ValueError(f"F0 không khớp quy tắc workload: {config}.")
        if config["n"] < config["m"] or config["first_trial"] != 1:
            raise ValueError(f"Tham số P0 không hợp lệ: {config}.")
        if config["s"] != compute_cvm_threshold(config["m"], config["epsilon"], config["delta"]):
            raise ValueError(f"s_reference không khớp công thức mã nguồn: {config}.")
        configs.append(config)

    random.Random(ORDER_SEED).shuffle(configs)
    return configs

In [ ]:
def new_run_dir(label: str) -> Path:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = EXPORT_DIR / f"{label}-{timestamp}-{uuid.uuid4().hex[:8]}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def require_native_ubuntu() -> list[str]:
    if platform.system() != "Linux":
        raise RuntimeError("Benchmark mới chỉ được chạy trên Linux native Ubuntu.")
    release = Path("/etc/os-release")
    fields = {}
    if release.is_file():
        for line in release.read_text(encoding="utf-8").splitlines():
            if "=" in line:
                key, value = line.split("=", 1)
                fields[key] = value.strip().strip('"')
    if fields.get("ID", "").lower() != "ubuntu":
        raise RuntimeError("Không xác nhận được Ubuntu qua /etc/os-release.")
    kernel_text = " ".join(
        (
            Path("/proc/version").read_text(encoding="utf-8", errors="replace"),
            Path("/proc/sys/kernel/osrelease").read_text(encoding="utf-8", errors="replace"),
        )
    )
    if (
        any(os.environ.get(name) for name in ("WSL_DISTRO_NAME", "WSL_INTEROP"))
        or "microsoft" in kernel_text.lower()
    ):
        raise RuntimeError("Phát hiện WSL; dừng benchmark Linux native.")
    detector = shutil.which("systemd-detect-virt")
    if detector is None:
        raise RuntimeError("Thiếu systemd-detect-virt; không xác minh được môi trường native.")
    evidence = [f"Ubuntu: {fields.get('PRETTY_NAME', 'Ubuntu')}", f"Kernel: {platform.release()}"]
    for option in ("--vm", "--container"):
        result = subprocess.run([detector, option], capture_output=True, text=True, check=False)
        detected = (result.stdout or result.stderr).strip() or "none"
        evidence.append(f"{option}: {detected}")
        if result.returncode == 1 and detected.lower() in {"", "none"}:
            continue
        if result.returncode == 0 and detected and detected.lower() != "none":
            raise RuntimeError(f"Phát hiện máy ảo/container ({detected}); dừng benchmark.")
        raise RuntimeError(
            f"Không xác minh được môi trường từ systemd-detect-virt {option}: "
            f"exit={result.returncode}, output={detected!r}."
        )
    return evidence


def path_for_display(path: Path) -> str:
    resolved = path.expanduser().resolve()
    try:
        return str(resolved.relative_to(ROOT.resolve()))
    except ValueError:
        try:
            return "~/" + str(resolved.relative_to(Path.home().resolve()))
        except ValueError:
            return str(resolved)


def load_existing_manifest(manifest_path: Path) -> tuple[dict, list[dict]]:
    manifest_path = manifest_path.expanduser().resolve()
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy manifest P0: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != 1 or manifest.get("dataset_id") != "timespace_linux_p0":
        raise ValueError("Manifest không thuộc bộ timespace_linux_p0 được hỗ trợ.")
    if manifest.get("classification") != "verified_real":
        raise ValueError("Manifest P0 phải được phân loại verified_real.")
    inputs = manifest.get("inputs")
    if not isinstance(inputs, list) or len(inputs) != manifest.get("group_count"):
        raise ValueError("Danh sách RAW/sidecar trong manifest không hợp lệ.")
    if (
        manifest.get("record_count"),
        manifest.get("group_count"),
        manifest.get("expected_plots"),
    ) != (135, 15, 3):
        raise ValueError("Manifest P0 phải mô tả 135 record, 15 nhóm và ba hình.")
    native_checks = manifest.get("native_linux_check")
    required_checks = {"--vm: none", "--container: none"}
    if (
        not isinstance(native_checks, list)
        or not any(check.startswith("Ubuntu:") for check in native_checks)
        or not required_checks.issubset(native_checks)
    ):
        raise ValueError("Manifest thiếu bằng chứng Ubuntu và kiểm tra VM/container.")

    dataset_root = manifest_path.parent.resolve()
    checked_inputs = []
    total_records = 0
    configurations = set()
    for item in inputs:
        if not isinstance(item, dict):
            raise ValueError("Mỗi đầu vào trong manifest phải là một đối tượng JSON.")
        raw_path = (dataset_root / item["raw"]).resolve()
        sidecar_path = (dataset_root / item["sidecar"]).resolve()
        for path in (raw_path, sidecar_path):
            try:
                path.relative_to(dataset_root)
            except ValueError as error:
                raise ValueError(
                    "Đường dẫn manifest phải nằm bên trong thư mục dữ liệu."
                ) from error
            if not path.is_file():
                raise FileNotFoundError(f"Thiếu tệp trong manifest: {path}")
        raw_sha256 = hashlib.sha256(raw_path.read_bytes()).hexdigest()
        sidecar_sha256 = hashlib.sha256(sidecar_path.read_bytes()).hexdigest()
        if raw_sha256 != item.get("raw_sha256") or sidecar_sha256 != item.get("sidecar_sha256"):
            raise ValueError(f"Checksum không khớp với manifest: {raw_path.name}")
        metadata = json.loads(sidecar_path.read_text(encoding="utf-8"))
        environments = [
            str(batch.get("environment", {}).get("os", "")) for batch in metadata.get("batches", [])
        ]
        if not environments or any(not value.lower().startswith("linux") for value in environments):
            raise ValueError(f"Sidecar không ghi môi trường Linux: {sidecar_path}")
        if int(metadata.get("records_count", -1)) != int(item.get("records", -2)):
            raise ValueError(f"Số record trong sidecar không khớp manifest: {sidecar_path}")
        key = (item.get("workload"), int(item.get("m", 0)))
        if key in configurations:
            raise ValueError(f"Cấu hình trùng trong manifest: {key}")
        configurations.add(key)
        total_records += int(item.get("records", 0))
        checked_inputs.append(
            {
                "raw": raw_path,
                "metadata": sidecar_path,
                "configuration": item.get("configuration"),
            }
        )

    expected = {(config["workload"], config["m"]): config for config in tv4_p0_configurations()}
    if configurations != set(expected):
        raise ValueError("Manifest không chứa đúng 15 cấu hình trong kế hoạch P0.")
    for item in inputs:
        config = expected[(item["workload"], int(item["m"]))]
        expected_values = {
            "n": config["n"],
            "F0": config["f0"],
            "epsilon": config["epsilon"],
            "delta": config["delta"],
            "trials": config["trials"],
            "first_trial": config["first_trial"],
            "seed_generator": config["seed_generator"],
            "seed_algorithm": config["seed_algorithm"],
        }
        for key, expected_value in expected_values.items():
            if item.get(key) != expected_value:
                raise ValueError(f"Tham số {key} trong manifest không khớp kế hoạch P0.")
        if int(item.get("records", 0)) != config["trials"]:
            raise ValueError("Số record của cấu hình không khớp số trial trong kế hoạch P0.")
    if total_records != manifest["record_count"]:
        raise ValueError("Tổng số record trong manifest không khớp.")
    return manifest, checked_inputs


def export_input_pair(raw_path: Path, metadata_path: Path, output_dir: Path, index: int):
    # Fresh benchmark pairs already live under this run directory.
    run_dir = output_dir.parent
    if raw_path.is_relative_to(run_dir) and metadata_path.is_relative_to(run_dir):
        return raw_path, metadata_path
    # Copy archived inputs so the exported report travels with the verified data.
    target = output_dir / "inputs" / f"dataset-{index:02d}"
    target.mkdir(parents=True)
    raw_copy = target / raw_path.name
    metadata_copy = target / metadata_path.name
    shutil.copyfile(raw_path, raw_copy)
    shutil.copyfile(metadata_path, metadata_copy)
    return raw_copy, metadata_copy


def write_run_report(
    path: Path, label: str, provenance: dict, overview: list[str], png_paths: tuple[Path, ...]
) -> None:
    lines = [
        f"# CVM-A1 Time/Space · {label}",
        "",
        f"{provenance['record_count']} bản ghi · {provenance['group_count']} nhóm · "
        f"{len(png_paths)} biểu đồ · `{provenance['classification']}`",
        "",
        "- [CSV tổng hợp](summary.csv)",
        "- [Danh mục RAW và sidecar](raw_index.csv)",
        "- [Nguồn gốc dữ liệu và môi trường](provenance.json)",
        "",
        "## Nguồn dữ liệu",
        "",
    ]
    for index, entry in enumerate(provenance["inputs"], start=1):
        raw_link = quote(entry["raw"], safe="/")
        sidecar_link = quote(entry["sidecar"], safe="/")
        detail_link = quote(entry["report"], safe="/")
        lines.append(
            f"- Bộ {index} · {entry['records']} bản ghi · [RAW]({raw_link}) · "
            f"[Sidecar]({sidecar_link}) · [Báo cáo TV4]({detail_link})"
        )
    lines.extend(("", "## Môi trường đo", ""))
    for entry in provenance["measurement_environments"]:
        lines.append(
            f"- {entry['os']} · GCC/Clang `{entry['compiler']}` · "
            f"Python {entry['python']} · CPU {entry['cpu']}"
        )
    lines.extend(f"- {check}" for check in provenance["native_linux_check"] or ())
    lines.extend(("", "## Biểu đồ", ""))
    lines.extend(f"- {note}" for note in overview)
    lines.extend(
        (
            "- Runtime thuật toán tính theo ms. Peak RSS là bộ nhớ thường trú cực đại "
            "của tiến trình C++, tính theo MiB trên hình và byte trong CSV. "
            "Kích thước tập mẫu tính theo số phần tử.",
            "- Các điểm đo mô tả cấu hình đã chạy. Kết luận Big-O cần phân tích "
            "chi phí thuật toán.",
        )
    )
    for png_path in png_paths:
        lines.extend(("", f"![{png_path.stem}]({quote(png_path.name)})"))
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def analyze_datasets(
    datasets: list[dict],
    output_dir: Path,
    *,
    label: str,
    native_evidence: list[str] | None = None,
    expected_records: int | None = None,
    expected_groups: int | None = None,
    source_manifest: Path | None = None,
) -> dict:
    from scripts.jupyter_analysis import analyze_jupyter_results
    from scripts.plot_results import plot_summary

    if not datasets:
        raise ValueError("Không có RAW/sidecar để phân tích.")
    output_dir = output_dir.resolve()
    raw_frames, summary_frames, provenance_rows, reports = [], [], [], []
    for index, item in enumerate(datasets, start=1):
        raw_path = Path(item["raw"]).expanduser().resolve()
        metadata_path = Path(item["metadata"]).expanduser().resolve()
        raw_path, metadata_path = export_input_pair(raw_path, metadata_path, output_dir, index)
        report = analyze_jupyter_results(
            raw_path, output_dir / "datasets", metadata_path=metadata_path
        )
        reports.append(report)
        raw_frames.append(report.raw)
        summary_frames.append(report.summary)
        provenance_rows.append(
            {
                "configuration": item.get("configuration", f"dataset-{index:02d}"),
                "raw": os.path.relpath(report.provenance.raw_path, output_dir),
                "sidecar": os.path.relpath(report.provenance.metadata_path, output_dir),
                "report": os.path.relpath(report.markdown_path, output_dir),
                "records": len(report.raw),
                "classification": report.provenance.classification,
                "raw_sha256": report.provenance.raw_sha256,
                "sidecar_sha256": report.provenance.metadata_sha256,
            }
        )

    raw = pd.concat(raw_frames, ignore_index=True)
    summary = pd.concat(summary_frames, ignore_index=True)
    if raw["run_id"].duplicated().any():
        raise ValueError("Phát hiện run_id trùng giữa các RAW đã xác thực.")
    if expected_records is not None and len(raw) != expected_records:
        raise ValueError(f"Cần {expected_records} record, nhận {len(raw)}.")
    if expected_groups is not None and len(summary) != expected_groups:
        raise ValueError(f"Cần {expected_groups} nhóm, nhận {len(summary)}.")

    summary_path = output_dir / "summary.csv"
    index_path = output_dir / "raw_index.csv"
    manifest_path = output_dir / "provenance.json"
    report_path = output_dir / "report.md"
    summary.to_csv(summary_path, index=False)
    index_frame = pd.DataFrame(provenance_rows)
    index_frame.to_csv(index_path, index=False)
    environment_fields = ("os", "python", "cpu", "compiler", "build_flags")
    environments = dict.fromkeys(
        tuple(batch["environment"].get(key, "—") for key in environment_fields)
        for report in reports
        for batch in report.provenance.metadata["batches"]
    )
    environment_rows = [
        dict(zip(environment_fields, values, strict=True)) for values in environments
    ]
    if len(reports) == 1:
        png_paths = tuple(
            Path(shutil.copyfile(path, output_dir / path.name)) for path in reports[0].png_paths
        )
    else:
        png_paths = tuple(plot_summary(summary.to_dict(orient="records"), output_dir, label=label))

    successful = raw.loc[raw["status"] == "success"]
    rss_count = int(successful["process_peak_memory_bytes"].count())
    overview = []
    if successful.empty:
        overview.append("Chưa có lượt thành công để vẽ biểu đồ.")
    elif not rss_count:
        overview.append("Không có số đo RSS")
    elif rss_count < len(successful):
        overview.append(f"Có số đo RSS ở {rss_count}/{len(successful)} lượt thành công.")
    if successful["m"].nunique() == 1:
        overview.append("Chỉ có một mức m. Chưa đủ điểm để nhận xét xu hướng.")

    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "analysis_api": "scripts.jupyter_analysis.analyze_jupyter_results",
        "plot_api": "scripts.plot_results.plot_summary",
        "classification": "verified_real",
        "kernel": sys.executable,
        "python": platform.python_version(),
        "kernel_os": platform.platform(),
        "packages": ENVIRONMENT["packages"],
        "notebook_sha256": hashlib.sha256(
            (ROOT / "notebooks" / "BIG_O_TimeSpace.ipynb").read_bytes()
        ).hexdigest(),
        "source_manifest": (
            {
                "path": path_for_display(source_manifest),
                "sha256": hashlib.sha256(source_manifest.read_bytes()).hexdigest(),
            }
            if source_manifest is not None
            else None
        ),
        "native_linux_check": native_evidence,
        "input_count": len(provenance_rows),
        "record_count": len(raw),
        "group_count": len(summary),
        "inputs": provenance_rows,
        "measurement_environments": environment_rows,
    }
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    write_run_report(report_path, label, manifest, overview, png_paths)

    status_table = (
        raw["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="records")
    )
    summary_columns = (
        "workload m n F0 epsilon delta s run_count success_count abort_count execution_error_count "
        "resource_failure_count halving_run_count runtime_median ram_median sample_median"
    ).split()
    display_summary = summary[summary_columns].rename(
        columns={
            "runtime_median": "Runtime median (ms)",
            "ram_median": "Peak RSS median (MiB)",
            "sample_median": "Peak sample median (phần tử)",
        }
    )
    rss_label = "Peak RSS median (MiB)"
    rss_mib = pd.to_numeric(display_summary[rss_label]).div(1024**2)
    display_summary[rss_label] = rss_mib.map(
        lambda value: f"{value:.4f}" if pd.notna(value) else "N/A"
    )
    display(Markdown(f"## Kết quả · {label}"))
    display(
        Markdown(
            f"**Phân loại TV4** `verified_real` · **RAW** {len(raw):,} bản ghi · "
            f"**Nhóm** {len(summary):,} · **Biểu đồ** {len(png_paths)}"
        )
    )
    display(Markdown("### Trạng thái lượt chạy"))
    display(status_table)
    display(Markdown("### RAW đã xác thực (10 dòng đầu)"))
    display(raw.head(10))
    display(Markdown("### Thống kê nhóm từ TV4"))
    display(display_summary)
    display(Markdown("### Provenance từng RAW/sidecar"))
    display(index_frame)
    display(Markdown("### Môi trường đo ghi trong sidecar"))
    display(pd.DataFrame(environment_rows))
    for note in overview:
        display(Markdown(f"- {note}"))
    for file_path in (report_path, summary_path, index_path, manifest_path):
        display(Markdown(f"**Tệp** `{path_for_display(file_path)}`"))
    for png_path in png_paths:
        display(Image(filename=str(png_path), width=900))
    return {"raw": raw, "summary": summary, "png_paths": png_paths, "provenance": provenance_rows}


def show_benchmark_result(result) -> None:
    display(
        Markdown(
            f"**API TV3** · {result.records_count} bản ghi · {result.configs_count} cấu hình\n\n"
            f"**RAW** `{path_for_display(result.csv_path)}`\n\n"
            f"**Sidecar** `{path_for_display(result.sidecar_path)}`"
        )
    )

In [ ]:
analysis_result = None
benchmark_results = []

if MODE == "READ_EXISTING_P0":
    source_manifest = READ_EXISTING_MANIFEST.expanduser().resolve()
    p0_manifest, checked_inputs = load_existing_manifest(source_manifest)
    display(
        Markdown(
            "Đang phân tích dữ liệu P0 Linux đã lưu.\n\n**Môi trường đo ghi trong manifest**\n\n"
            + "\n".join(f"- {value}" for value in p0_manifest["native_linux_check"])
        )
    )
    run_dir = new_run_dir("linux-p0-analysis")
    analysis_result = analyze_datasets(
        checked_inputs,
        run_dir / "analysis",
        label="P0 Linux đã lưu",
        native_evidence=p0_manifest.get("native_linux_check"),
        expected_records=p0_manifest["record_count"],
        expected_groups=p0_manifest["group_count"],
        source_manifest=source_manifest,
    )

elif MODE == "SMOKE_LINUX":
    native_evidence = require_native_ubuntu()
    display(
        Markdown(
            "**Xác nhận môi trường:**\n\n" + "\n".join(f"- {item}" for item in native_evidence)
        )
    )
    from src.jupyter_adapter import run_jupyter_benchmark

    run_dir = new_run_dir("smoke-linux")
    result = run_jupyter_benchmark(preset="smoke", output_dir=run_dir / "real", verbose=False)
    benchmark_results.append(result)
    show_benchmark_result(result)
    analysis_result = analyze_datasets(
        [{"raw": result.csv_path, "metadata": result.sidecar_path, "configuration": "SMOKE_LINUX"}],
        run_dir / "analysis",
        label="SMOKE_LINUX",
        native_evidence=native_evidence,
    )

elif MODE == "CUSTOM_SWEEP":
    if not RUN_CUSTOM:
        if CUSTOM_PROFILE == "TV4_P0":
            plan = tv4_p0_configurations()
        else:
            plan = [
                {
                    "m": m,
                    "n": CUSTOM_N,
                    "f0": CUSTOM_F0,
                    "workload": workload,
                    "epsilon": CUSTOM_EPSILON,
                    "delta": CUSTOM_DELTA,
                    "trials": CUSTOM_TRIALS,
                    "seed_generator": CUSTOM_SEED_GENERATOR,
                    "seed_algorithm": CUSTOM_SEED_ALGORITHM,
                }
                for workload in CUSTOM_WORKLOADS
                for m in CUSTOM_M
            ]
        attempts = sum(config["trials"] for config in plan)
        display(
            Markdown(
                f"**Xem trước** · {len(plan)} cấu hình · {attempts} lượt dự kiến. "
                "Đặt `RUN_CUSTOM = True` để chạy benchmark trên Ubuntu native."
            )
        )
        preview = pd.DataFrame(plan)
        if CUSTOM_PROFILE == "TV4_P0":
            preview["F0_over_s"] = preview["f0"] / preview["s"]
        display(preview)
    else:
        native_evidence = require_native_ubuntu()
        display(
            Markdown(
                "**Xác nhận môi trường:**\n\n" + "\n".join(f"- {item}" for item in native_evidence)
            )
        )
        from src.jupyter_adapter import run_jupyter_benchmark

        run_dir = new_run_dir("linux-p0" if CUSTOM_PROFILE == "TV4_P0" else "custom-linux")
        if CUSTOM_PROFILE == "TV4_P0":
            datasets = []
            configs = tv4_p0_configurations()
            for index, config in enumerate(configs, start=1):
                tag = f"{config['workload']}_m{config['m']}"
                print(
                    f"TV3 cấu hình {index}/{len(configs)} · {tag} · "
                    f"F0={config['f0']} · {config['trials']} trial",
                    flush=True,
                )
                result = run_jupyter_benchmark(
                    m=config["m"],
                    n=config["n"],
                    f0=config["f0"],
                    epsilon=config["epsilon"],
                    delta=config["delta"],
                    workload=config["workload"],
                    trials=config["trials"],
                    seed_generator=config["seed_generator"],
                    seed_algorithm=config["seed_algorithm"],
                    output_dir=run_dir / "real",
                    tag=tag,
                    verbose=False,
                )
                benchmark_results.append(result)
                datasets.append(
                    {"raw": result.csv_path, "metadata": result.sidecar_path, "configuration": tag}
                )
            total_records = sum(item.records_count for item in benchmark_results)
            display(
                Markdown(
                    f"**API TV3** · {total_records} bản ghi · {len(configs)} cấu hình "
                    f"· {len(benchmark_results)} cặp RAW/sidecar"
                )
            )
            analysis_result = analyze_datasets(
                datasets,
                run_dir / "analysis",
                label="Linux P0",
                native_evidence=native_evidence,
                expected_records=135,
                expected_groups=15,
            )
        else:
            result = run_jupyter_benchmark(
                m=CUSTOM_M,
                n=CUSTOM_N,
                f0=CUSTOM_F0,
                epsilon=CUSTOM_EPSILON,
                delta=CUSTOM_DELTA,
                workload=CUSTOM_WORKLOADS,
                trials=CUSTOM_TRIALS,
                seed_generator=CUSTOM_SEED_GENERATOR,
                seed_algorithm=CUSTOM_SEED_ALGORITHM,
                output_dir=run_dir / "real",
                tag="custom_sweep",
                verbose=False,
            )
            benchmark_results.append(result)
            show_benchmark_result(result)
            analysis_result = analyze_datasets(
                [
                    {
                        "raw": result.csv_path,
                        "metadata": result.sidecar_path,
                        "configuration": "MANUAL_SWEEP",
                    }
                ],
                run_dir / "analysis",
                label="CUSTOM_SWEEP",
                native_evidence=native_evidence,
            )

## Diễn giải và giới hạn

RAW và sidecar được giữ nguyên theo từng cấu hình. TV4 xác thực từng cặp trước khi gộp bảng và vẽ runtime lõi, peak RSS tiến trình C++ và peak sample. RSS không bao gồm bộ nhớ của Python/Jupyter.

Các bảng và hình mô tả những cấu hình đã đo. Chín lượt mỗi cấu hình theo giao thức P0 không tạo ra bảo đảm về độ chính xác thống kê. Kết luận về Big-O cần dựa thêm vào phân tích chi phí thuật toán; không suy rộng từ các điểm đo hữu hạn.